# 라틴 하이퍼큐브 표본 실습

**Latin Hypercube Sampling · LHS**

각 변수 구간을 고르게 덮도록 표본을 배치해 적은 표본으로 공간을 넓게 살피는 표본추출법.

소재 분야에서 이해하기: 5개 공정 변수 공간을 20점으로 고르게 탐색한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SciPy 준몬테카를로 문서](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)

## 1. 무작위 표본과 LHS 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from scipy.stats import qmc

size, dimension = 20, 2
random_points = rng.random((size, dimension))
lhs_points = qmc.LatinHypercube(dimension, seed=0).random(size)
sobol_points = qmc.Sobol(dimension, scramble=True, seed=0).random(size)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for axis, (name, points) in zip(axes, [('random', random_points), ('Latin hypercube', lhs_points),
                                       ('Sobol sequence', sobol_points)]):
    axis.scatter(points[:, 0], points[:, 1], s=30)
    for line in np.linspace(0, 1, size + 1):
        axis.axvline(line, lw=0.3, color='gray'); axis.axhline(line, lw=0.3, color='gray')
    axis.set_title(name); axis.set_xlim(0, 1); axis.set_ylim(0, 1)
plt.tight_layout(); plt.show()

In [ ]:
for name, points in [('random', random_points), ('Latin hypercube', lhs_points), ('Sobol', sobol_points)]:
    # 각 변수의 구간을 몇 개나 덮었는지 (LHS 는 모든 구간을 정확히 한 번씩 덮습니다)
    covered = [len(set((points[:, d] * size).astype(int))) for d in range(dimension)]
    print('%-16s 변수별 덮은 구간 수 %s / %d,  불일치도 %.4f'
          % (name, covered, size, qmc.discrepancy(points)))

## 2. 표본 배치가 대리 모델 정확도를 바꿉니다

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

def truth(points):
    return np.sin(3 * points[:, 0]) + points[:, 1] ** 2

test_points = rng.random((3000, 2))
for name, points in [('random', random_points), ('Latin hypercube', lhs_points), ('Sobol', sobol_points)]:
    model = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(0.3),
                                     normalize_y=True, random_state=0).fit(points, truth(points))
    error = np.mean(np.abs(model.predict(test_points) - truth(test_points)))
    print('%-16s 20점 학습 후 평균 절대 오차 %.4f' % (name, error))
print('\n표본 하나가 비싼 실험이라면, 어디를 볼지 정하는 것이 모델 선택보다 중요할 수 있습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#latin-hypercube)을 여세요.